# Notebook 13 — LLM Scaling Study: Jaccard Similarity Curve

**Purpose:** Compare pipeline NER systems (spaCy / Stanza / Flair) against LLMs of varying
scale using Jaccard similarity — no fixed gold standard.
Addresses gold-standard bias identified in nb05/nb07.

| | |
|---|---|
| **Existing results (loaded)** | Qwen 2.5 7B EN (nb04) · Llama 3.3 70B EN (nb09) |
| **New Groq models** | Llama 3.1 8B · GPT-OSS 20B · Qwen3 32B · GPT-OSS 120B |
| **Checkpoint format** | Old checkpoints (Qwen7B, Llama70B): flat `{article_id: {...}}` dict · New checkpoints: `{results: [...]}` list |
| **Checkpoint policy** | Saved every 10 articles. Stops cleanly on any API error. Re-running Cell 7 resumes. |


In [ ]:
# ── CELL 1 : INSTALLATION ────────────────────────────────────────────────────
!pip install -q groq
!pip install -q 'numpy>=2.0'   # must be last


In [ ]:
# ── CELL 2 : IMPORTS & CONFIGURATION ─────────────────────────────────────────

import os, json, pickle, time, re, traceback
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=False)

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT        = Path('/content/drive/MyDrive/thesis')
DATA_PROC   = ROOT / 'Project/Data/Processed'
FIGURES_DIR = ROOT / 'Project/Outputs/Figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── API token ─────────────────────────────────────────────────────────────────
GROQ_TOKEN = userdata.get('GROQ_TOKEN')

# ── Checkpointing ─────────────────────────────────────────────────────────────
SAVE_EVERY               = 10
MAX_CONSECUTIVE_FAILURES = 5
MAX_FAILURE_RATE         = 0.20
GROQ_SLEEP_BETWEEN       = 2.0
RETRY_DELAYS             = [4, 8, 16]

# ── Target models ─────────────────────────────────────────────────────────────
GROQ_TARGET_MODELS = {
    "llama-3.1-8b-instant" : {"scale_B": 8,   "family": "llama", "input": "en"},
    "openai/gpt-oss-20b"   : {"scale_B": 20,  "family": "gpt",   "input": "en"},
    "qwen/qwen3-32b"       : {"scale_B": 32,  "family": "qwen3", "input": "en"},
    "openai/gpt-oss-120b"  : {"scale_B": 120, "family": "gpt",   "input": "en"},
}

# ── NER prompt — same as nb04/nb09 ───────────────────────────────────────────
NER_SYSTEM_PROMPT = """You are a named entity recognition (NER) system.
Extract ALL named entities from the text provided.

Return ONLY a valid JSON object — no markdown, no explanation, no preamble:
{
  "entities": [
    {"text": "<entity surface form>", "label": "<TYPE>"}
  ]
}

Use exactly these four labels:
  PER   -- person names
  LOC   -- locations, countries, cities, geographical features
  ORG   -- organisations, companies, institutions
  MISC  -- other named entities (events, products, languages, nationalities, etc.)

If no entities are found return:  {"entities": []}"""

NER_USER_TEMPLATE = "Extract named entities from the following text:\n\n{text}"

print("✓ Configuration loaded")
print(f"  Groq token : {'SET' if GROQ_TOKEN else '⚠ MISSING'}")
print(f"  Models     : {list(GROQ_TARGET_MODELS.keys())}")


In [ ]:
# ── CELL 3 : CHECK AVAILABLE GROQ MODELS ─────────────────────────────────────

from groq import Groq
groq_client = Groq(api_key=GROQ_TOKEN)

print("Querying Groq for live models...")
try:
    available_groq = {m.id for m in groq_client.models.list().data}
    print(f"  {len(available_groq)} models available\n")
except Exception as e:
    available_groq = set()
    print(f"  ⚠ Could not list models: {e}")

confirmed, skipped = {}, {}
for model_id, meta in GROQ_TARGET_MODELS.items():
    if model_id in available_groq:
        confirmed[model_id] = meta
        print(f"  ✓ CONFIRMED : {model_id}  ({meta['scale_B']}B)")
    else:
        skipped[model_id] = meta
        print(f"  ✗ NOT FOUND : {model_id}  — will skip")


In [ ]:
# ── CELL 4a : LOAD EXISTING DATA ─────────────────────────────────────────────
# Handles the actual column names and checkpoint formats in this project.
#
# Checkpoint format discovery (from diagnostic):
#   OLD (Qwen 7B, Llama 70B) : flat dict  {article_id: {entities, status, ...}}
#   NEW (nb13 runs)           : list dict  {results: [{article_id, entities, status}, ...]}
#
# Pipeline column names (from diagnostic):
#   ner_spacy, ner_stanza, ner_flair   (NOT spacy_entities etc.)
#   content_en                          (NOT translated_text)

print("Loading pipeline results...")
pipeline_df = pd.read_pickle(DATA_PROC / 'ner_pipeline_results.pkl')
print(f"  shape   : {pipeline_df.shape}")

# ── Correct column names ──────────────────────────────────────────────────────
PIPELINE_ENTITY_COLS = {
    'spaCy'  : 'ner_spacy',
    'Stanza' : 'ner_stanza',
    'Flair'  : 'ner_flair',
}
TEXT_COL = 'content_en'

for name, col in PIPELINE_ENTITY_COLS.items():
    status = "✓ OK" if col in pipeline_df.columns else "⚠ MISSING"
    print(f"  {name}: '{col}' — {status}")
status = "✓ OK" if TEXT_COL in pipeline_df.columns else "⚠ MISSING"
print(f"  text  : '{TEXT_COL}' — {status}")

# ── Existing LLM checkpoints ──────────────────────────────────────────────────
EXISTING_LLM_FILES = {
    "qwen-2.5-7b-instruct"    : {"file"    : DATA_PROC / 'ner_llm_checkpoint.pkl',
                                  "scale_B" : 7,  "family": "qwen",  "input": "en"},
    "llama-3.3-70b-versatile" : {"file"    : DATA_PROC / 'ner_llama70b_checkpoint.pkl',
                                  "scale_B" : 70, "family": "llama", "input": "en"},
}

existing_llm_results = {}
for model_id, info in EXISTING_LLM_FILES.items():
    fpath = info["file"]
    if fpath.exists():
        with open(fpath, 'rb') as f:
            raw = pickle.load(f)
        existing_llm_results[model_id] = raw
        # Old format: flat dict {article_id: {status, entities, ...}}
        n_ok  = sum(1 for v in raw.values()
                    if isinstance(v, dict) and v.get('status') != 'api_error')
        n_err = sum(1 for v in raw.values()
                    if isinstance(v, dict) and v.get('status') == 'api_error')
        print(f"  ✓ Loaded {model_id}: {n_ok} ok, {n_err} api_error")
    else:
        print(f"  ⚠ NOT FOUND: {fpath.name}")

# ── ARTICLE_IDS_183 — from Qwen 7B ok articles ───────────────────────────────
# Old checkpoint is a flat dict; filter out api_error entries.
if "qwen-2.5-7b-instruct" in existing_llm_results:
    qwen_ckpt = existing_llm_results["qwen-2.5-7b-instruct"]
    ARTICLE_IDS_183 = [
        aid for aid, data in qwen_ckpt.items()
        if isinstance(data, dict) and data.get('status') != 'api_error'
    ]
else:
    # Fallback: use all article_ids from pipeline_df
    ARTICLE_IDS_183 = list(pipeline_df['article_id'].unique())

print(f"\n  Working set: {len(ARTICLE_IDS_183)} articles")

# ── article_id -> text lookup ─────────────────────────────────────────────────
if TEXT_COL in pipeline_df.columns:
    id_to_text = dict(zip(pipeline_df['article_id'], pipeline_df[TEXT_COL]))
    print(f"  ✓ Text lookup ready ({len(id_to_text)} entries)")
else:
    id_to_text = {}
    print(f"  ⚠ Text column '{TEXT_COL}' missing — check pipeline_df columns")


In [ ]:
# ── CELL 4b : CLEAR BAD CHECKPOINTS ─────────────────────────────────────────
# GPT-OSS 20B, Qwen3 32B, GPT-OSS 120B ran with ARTICLE_IDS_183=0 in the
# previous run — they completed instantly with no data.
# Llama 8B ran correctly (183 articles) — its checkpoint is preserved.

def ckpt_path(model_id: str) -> Path:
    safe_id = re.sub(r'[/:\\]', '_', model_id)
    return DATA_PROC / f'nb13_{safe_id}_checkpoint.pkl'

MODELS_TO_CLEAR = [
    "openai/gpt-oss-20b",   # ran with 0 articles — empty checkpoint
    "qwen/qwen3-32b",       # ran with 0 articles — empty checkpoint
    "openai/gpt-oss-120b",  # ran with 0 articles — empty checkpoint
]

print("Clearing checkpoints that ran with 0 articles...")
for model_id in MODELS_TO_CLEAR:
    fp = ckpt_path(model_id)
    if fp.exists():
        os.remove(fp)
        print(f"  Cleared : {fp.name}")
    else:
        print(f"  Already clean : {fp.name}")

# Confirm Llama 8B is intact and has real data
llama8_fp = ckpt_path("llama-3.1-8b-instant")
if llama8_fp.exists():
    with open(llama8_fp, 'rb') as f:
        llama8_ckpt = pickle.load(f)
    n_ok = len([r for r in llama8_ckpt.get('results', []) if r.get('status') == 'ok'])
    print(f"\n  ✓ Preserved : llama-3.1-8b-instant ({n_ok} articles — will not rerun)")
else:
    print(f"\n  – Llama 8B checkpoint not found — will run from scratch")

print(f"\n  ARTICLE_IDS_183 = {len(ARTICLE_IDS_183)} articles — ready for new runs")


In [ ]:
# ── CELL 5 : CHECKPOINT UTILITIES ────────────────────────────────────────────

def load_checkpoint(model_id: str) -> dict:
    fp = ckpt_path(model_id)
    if fp.exists():
        with open(fp, 'rb') as f:
            ckpt = pickle.load(f)
        done = len([r for r in ckpt['results'] if r['status'] == 'ok'])
        fail = len(ckpt['failed_ids'])
        print(f"  ↺ Resuming {model_id}: {done} done, {fail} failed previously")
        return ckpt
    return {
        'model_id'   : model_id,
        'results'    : [],
        'failed_ids' : [],
        'stop_reason': None,
        'started_at' : datetime.now().isoformat(),
        'updated_at' : None,
    }

def save_checkpoint(ckpt: dict, model_id: str):
    ckpt['updated_at'] = datetime.now().isoformat()
    with open(ckpt_path(model_id), 'wb') as f:
        pickle.dump(ckpt, f)

def is_complete(ckpt: dict) -> bool:
    return ckpt.get('stop_reason') == 'complete'

def processed_ids(ckpt: dict) -> set:
    return {r['article_id'] for r in ckpt['results']} | set(ckpt['failed_ids'])

print("✓ Checkpoint utilities ready")


In [ ]:
# ── CELL 6 : EXTRACTION FUNCTIONS ────────────────────────────────────────────

def _parse_entities(raw: str) -> list:
    """
    Parse JSON entity list from model response.
    Handles: Qwen3 think blocks (closed + truncated), GPT-OSS truncation,
    empty strings, markdown fences. Returns [] on any failure.
    """
    raw = raw.strip()
    if not raw:
        return []
    if '<think>' in raw:
        if '</think>' in raw:
            raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
        else:
            brace_pos = raw.rfind('{')
            raw = raw[brace_pos:] if brace_pos != -1 else ''
    raw = re.sub(r'^```(?:json)?\s*', '', raw, flags=re.MULTILINE)
    raw = re.sub(r'\s*```$',          '', raw, flags=re.MULTILINE)
    raw = raw.strip()
    try:
        data     = json.loads(raw)
        entities = data.get('entities', [])
        return [e for e in entities
                if isinstance(e, dict) and 'text' in e and 'label' in e]
    except json.JSONDecodeError:
        return []


def build_messages(model_id: str, text: str) -> tuple:
    """
    Returns (messages_list, max_tokens) per model family.
    GPT-OSS : merge system+user into single user message (system role ignored)
    Qwen3   : /no_think prefix + 2048 token budget
    Others  : standard system + user, 512 tokens
    """
    user_text = NER_USER_TEMPLATE.format(text=text[:3000])

    if 'gpt-oss' in model_id.lower():
        messages   = [{"role": "user",
                       "content": NER_SYSTEM_PROMPT + "\n\n" + user_text}]
        max_tokens = 2048

    elif 'qwen3' in model_id.lower():
        messages   = [{"role": "system", "content": NER_SYSTEM_PROMPT},
                      {"role": "user",   "content": "/no_think\n\n" + user_text}]
        max_tokens = 2048

    else:
        messages   = [{"role": "system", "content": NER_SYSTEM_PROMPT},
                      {"role": "user",   "content": user_text}]
        max_tokens = 512

    return messages, max_tokens


def extract_ner_groq(client, model_id: str, text: str):
    """
    Call Groq. Returns (entity_list, raw_str).
    Raises RuntimeError on unrecoverable error.
    """
    from groq import RateLimitError, APITimeoutError, APIStatusError

    messages, max_tokens = build_messages(model_id, text)
    last_exc = None

    for attempt, delay in enumerate([0] + RETRY_DELAYS, start=1):
        if delay:
            print(f"    Waiting {delay}s before retry {attempt}...")
            time.sleep(delay)
        try:
            resp = client.chat.completions.create(
                model=model_id, messages=messages,
                max_tokens=max_tokens, temperature=0.0,
            )
            raw = resp.choices[0].message.content or ''
            return _parse_entities(raw), raw

        except RateLimitError as e:
            last_exc = e
            if attempt <= len(RETRY_DELAYS):
                print(f"    429 rate-limit (attempt {attempt}), backing off...")
            else:
                raise RuntimeError(f"RATE_LIMIT after {attempt} attempts: {e}") from e
        except APITimeoutError as e:
            last_exc = e
            if attempt <= len(RETRY_DELAYS):
                print(f"    Timeout (attempt {attempt}), retrying...")
            else:
                raise RuntimeError(f"TIMEOUT after {attempt} attempts: {e}") from e
        except APIStatusError as e:
            raise RuntimeError(f"API_STATUS_{e.status_code}: {e.message}") from e
        except Exception as e:
            raise RuntimeError(f"UNEXPECTED: {e}") from e

    raise RuntimeError(f"Exhausted retries. Last: {last_exc}")


print("✓ Extraction functions ready")
print("  _parse_entities : think blocks · truncation · fences · empty strings")
print("  build_messages  : GPT-OSS (merged+2048) · Qwen3 (no_think+2048) · Llama (std+512)")


In [ ]:
# ── CELL 7 : RUN LOOP WITH CHECKPOINTING ─────────────────────────────────────
# Re-running is always safe — resumes from last checkpoint.
# Stops and saves on: rate_limit / consecutive_failures / high_failure_rate.

def run_model(model_id, extract_fn, article_ids, id_to_text,
              sleep_between=GROQ_SLEEP_BETWEEN):

    ckpt = load_checkpoint(model_id)

    if is_complete(ckpt):
        n_ok = len([r for r in ckpt['results'] if r['status'] == 'ok'])
        print(f"  ✓ {model_id} already complete ({n_ok} articles) — skipping")
        return ckpt

    already_done = processed_ids(ckpt)
    remaining    = [aid for aid in article_ids if aid not in already_done]

    if not remaining:
        ckpt['stop_reason'] = 'complete'
        save_checkpoint(ckpt, model_id)
        print(f"  ✓ {model_id} — all articles done, marked complete")
        return ckpt

    print(f"\n{'='*64}")
    print(f"  MODEL   : {model_id}")
    print(f"  Pending : {len(remaining)}  (done: {len(already_done)})")
    print(f"  Stops   : {MAX_CONSECUTIVE_FAILURES} consecutive failures "
          f"OR >{MAX_FAILURE_RATE*100:.0f}% failure rate")
    print(f"{'='*64}")

    consec_fail = 0
    total_ok    = len([r for r in ckpt['results'] if r['status'] == 'ok'])
    total_fail  = len(ckpt['failed_ids'])

    for i, article_id in enumerate(remaining, start=1):
        text = id_to_text.get(article_id, '')

        if not text:
            ckpt['failed_ids'].append(article_id)
            consec_fail += 1
            total_fail  += 1
            print(f"  [{i}/{len(remaining)}] {article_id} — NO TEXT")
        else:
            try:
                entities, raw = extract_fn(text)
                ckpt['results'].append({
                    'article_id'   : article_id,
                    'entities'     : entities,
                    'raw_response' : raw,
                    'status'       : 'ok',
                })
                consec_fail  = 0
                total_ok    += 1
                if i % 20 == 0 or i == len(remaining):
                    print(f"  [{i}/{len(remaining)}]  ok={total_ok}  "
                          f"fail={total_fail}  last_entities={len(entities)}")
                time.sleep(sleep_between)

            except RuntimeError as exc:
                err_msg     = str(exc)
                consec_fail += 1
                total_fail  += 1
                ckpt['failed_ids'].append(article_id)
                print(f"\n  ⚠ [{i}/{len(remaining)}] FAILED [{article_id}]: {err_msg}")

                stop_reason = None
                if 'RATE_LIMIT' in err_msg or 'API_STATUS_402' in err_msg:
                    stop_reason = 'rate_limit'
                    print(f"\n  ✋ STOPPING — rate limit / credit exhaustion.")
                    print(f"     Re-run this cell to resume.")
                elif consec_fail >= MAX_CONSECUTIVE_FAILURES:
                    stop_reason = 'consecutive_failures'
                    print(f"\n  ✋ STOPPING — {consec_fail} consecutive failures.")
                    print(f"     Check API status then re-run.")
                else:
                    total_seen = total_ok + total_fail
                    if total_seen >= 20 and (total_fail / total_seen) > MAX_FAILURE_RATE:
                        stop_reason = 'high_failure_rate'
                        rate = total_fail / total_seen * 100
                        print(f"\n  ✋ STOPPING — failure rate {rate:.1f}%.")

                if stop_reason:
                    ckpt['stop_reason'] = stop_reason
                    save_checkpoint(ckpt, model_id)
                    print(f"     Checkpoint: {ckpt_path(model_id).name}")
                    print(f"     Progress  : {total_ok} ok / {total_fail} failed / "
                          f"{len(remaining)-i} remaining")
                    return ckpt

        if i % SAVE_EVERY == 0:
            save_checkpoint(ckpt, model_id)
            print(f"  💾 checkpoint saved at article {i}")

    ckpt['stop_reason'] = 'complete'
    save_checkpoint(ckpt, model_id)
    print(f"\n  ✅ COMPLETE: {model_id}  (ok={total_ok}  failed={total_fail})")
    return ckpt


# ── Run all confirmed Groq models ─────────────────────────────────────────────
nb13_new_results = {}

for model_id in confirmed:
    fn = (lambda text, mid=model_id: extract_ner_groq(groq_client, mid, text))
    nb13_new_results[model_id] = run_model(
        model_id=model_id, extract_fn=fn,
        article_ids=ARTICLE_IDS_183, id_to_text=id_to_text,
        sleep_between=GROQ_SLEEP_BETWEEN,
    )


In [ ]:
# ── CELL 8 : CONSOLIDATE ALL LLM RESULTS ─────────────────────────────────────

LABEL_NORM = {
    "PERSON": "PER", "person": "PER",
    "LOCATION": "LOC", "location": "LOC", "GPE": "LOC", "gpe": "LOC", "FAC": "LOC",
    "ORGANIZATION": "ORG", "organisation": "ORG", "organization": "ORG",
    "MISCELLANEOUS": "MISC", "miscellaneous": "MISC",
    "EVENT": "MISC", "PRODUCT": "MISC", "LANGUAGE": "MISC", "NORP": "MISC",
    "WORK_OF_ART": "MISC", "LAW": "MISC", "DATE": "MISC", "TIME": "MISC",
    "PER": "PER", "LOC": "LOC", "ORG": "ORG", "MISC": "MISC",
}
VALID_LABELS = {"PER", "LOC", "ORG", "MISC"}

def normalize_entities(entity_list) -> set:
    """Convert any entity representation to set of (text, label) tuples."""
    out = set()
    if not entity_list:
        return out
    for e in entity_list:
        # Handle dict format: {text, label, ...}
        if isinstance(e, dict):
            text  = str(e.get('text', '')).strip().lower()
            label = LABEL_NORM.get(str(e.get('label', '')), None)
        # Handle tuple/list format: (text, label) or [text, label]
        elif isinstance(e, (list, tuple)) and len(e) >= 2:
            text  = str(e[0]).strip().lower()
            label = LABEL_NORM.get(str(e[1]), None)
        else:
            continue
        if text and label in VALID_LABELS:
            out.add((text, label))
    return out


def build_entity_lookup(ckpt) -> dict:
    """
    Build {article_id: set_of_(text,label)} from either checkpoint format.

    OLD format (Qwen7B, Llama70B) : flat dict {article_id: {entities, status, ...}}
    NEW format (nb13 runs)        : {results: [{article_id, entities, status}, ...]}
    """
    if isinstance(ckpt, dict) and 'results' in ckpt:
        # New format
        return {
            r['article_id']: normalize_entities(r.get('entities', []))
            for r in ckpt.get('results', [])
            if r.get('status') == 'ok'
        }
    else:
        # Old flat dict format — skip api_error entries
        return {
            aid: normalize_entities(data.get('entities', []))
            for aid, data in ckpt.items()
            if isinstance(data, dict) and data.get('status') != 'api_error'
        }


# ── Merge all LLM results ─────────────────────────────────────────────────────
all_llm_lookups = {}
all_llm_meta    = {}

for model_id, info in EXISTING_LLM_FILES.items():
    if model_id in existing_llm_results:
        lookup = build_entity_lookup(existing_llm_results[model_id])
        all_llm_lookups[model_id] = lookup
        all_llm_meta[model_id]    = {k: v for k, v in info.items() if k != 'file'}
        print(f"  ✓ {model_id:<44} {len(lookup)} articles")

for model_id, ckpt in nb13_new_results.items():
    n_ok = len([r for r in ckpt['results'] if r['status'] == 'ok'])
    if n_ok > 0:
        all_llm_lookups[model_id] = build_entity_lookup(ckpt)
        all_llm_meta[model_id]    = confirmed.get(model_id, {})
        print(f"  ✓ {model_id:<44} {n_ok} articles")
    else:
        print(f"  ✗ {model_id:<44} 0 ok — excluded from Jaccard")

print(f"\n  Total LLM systems: {len(all_llm_lookups)}")

# ── Pipeline entity lookups ───────────────────────────────────────────────────
# Pipeline columns (ner_spacy, ner_stanza, ner_flair) may store entities as:
#   list of dicts    [{text, label, start, end}, ...]
#   list of tuples   [(text, label), ...]
#   empty list       []
# normalize_entities handles all of these.

pipeline_lookups = {}
article_id_set   = set(ARTICLE_IDS_183)

for name, col in PIPELINE_ENTITY_COLS.items():
    if col not in pipeline_df.columns:
        print(f"  ⚠ Skipping {name}: column '{col}' missing")
        continue
    lookup = {}
    for _, row in pipeline_df.iterrows():
        aid = row['article_id']
        if aid not in article_id_set:
            continue
        raw_ents = row[col]
        lookup[aid] = normalize_entities(raw_ents) if isinstance(raw_ents, (list, tuple)) else set()
    pipeline_lookups[name] = lookup
    print(f"  ✓ Pipeline {name:<10} {len(lookup)} articles")

print("\n✓ All lookups consolidated")


In [ ]:
# ── CELL 9 : COMPUTE JACCARD SIMILARITY ──────────────────────────────────────

def jaccard(set_a: set, set_b: set) -> float:
    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

def compute_jaccard_stats(p_lookup, l_lookup, article_ids) -> dict:
    scores = [
        jaccard(p_lookup.get(aid, set()), l_lookup.get(aid, set()))
        for aid in article_ids
        if aid in p_lookup and aid in l_lookup
    ]
    return {
        'mean'  : float(np.mean(scores))   if scores else float('nan'),
        'median': float(np.median(scores)) if scores else float('nan'),
        'std'   : float(np.std(scores))    if scores else float('nan'),
        'n'     : len(scores),
    }

llm_order = sorted(all_llm_meta, key=lambda m: all_llm_meta[m].get('scale_B', 0))

jaccard_records = []
for pipeline_name, p_lookup in pipeline_lookups.items():
    for llm_id in llm_order:
        if llm_id not in all_llm_lookups:
            continue
        stats = compute_jaccard_stats(
            p_lookup, all_llm_lookups[llm_id], ARTICLE_IDS_183)
        jaccard_records.append({
            'pipeline': pipeline_name,
            'llm'     : llm_id,
            'scale_B' : all_llm_meta[llm_id].get('scale_B', 0),
            'family'  : all_llm_meta[llm_id].get('family', '?'),
            **stats,
        })

jaccard_df = pd.DataFrame(jaccard_records)

print("Jaccard similarity matrix (mean):")
pivot     = jaccard_df.pivot(index='pipeline', columns='llm', values='mean').round(3)
col_order = [m for m in llm_order if m in pivot.columns]
print(pivot[col_order].to_string())

jaccard_df.to_csv(DATA_PROC / 'nb13_jaccard_matrix.csv', index=False)
print(f"\n✓ Saved → nb13_jaccard_matrix.csv")


In [ ]:
# ── CELL 10 : SCALING CURVE PLOT ─────────────────────────────────────────────

PIPELINE_STYLES = {
    'Flair' : {'color': '#1f77b4', 'marker': 'o', 'lw': 2.2, 'ms': 9},
    'Stanza': {'color': '#ff7f0e', 'marker': 's', 'lw': 2.2, 'ms': 9},
    'spaCy' : {'color': '#2ca02c', 'marker': '^', 'lw': 2.2, 'ms': 9},
}

def x_label(model_id: str, scale_B: float) -> str:
    s = int(scale_B)
    if 'qwen-2.5' in model_id : return f"Qwen 2.5\n{s}B"
    if 'qwen3'   in model_id  : return f"Qwen3\n{s}B"
    if 'gpt-oss' in model_id  : return f"GPT-OSS\n{s}B"
    if '8b'      in model_id  : return f"Llama\n{s}B"
    if '70b'     in model_id  : return f"Llama\n{s}B"
    return f"{s}B"

fig, ax = plt.subplots(figsize=(11, 6))

for pipeline_name, style in PIPELINE_STYLES.items():
    sub = jaccard_df[jaccard_df['pipeline'] == pipeline_name].sort_values('scale_B')
    if sub.empty:
        continue
    ax.plot(sub['scale_B'], sub['mean'],
            label=pipeline_name,
            color=style['color'], marker=style['marker'],
            linewidth=style['lw'], markersize=style['ms'], zorder=3)
    ax.fill_between(sub['scale_B'],
                    sub['mean'] - sub['std'],
                    sub['mean'] + sub['std'],
                    alpha=0.12, color=style['color'])

scale_rows = jaccard_df.drop_duplicates('scale_B').sort_values('scale_B')
ax.set_xticks(scale_rows['scale_B'].tolist())
ax.set_xticklabels(
    [x_label(r['llm'], r['scale_B']) for _, r in scale_rows.iterrows()],
    fontsize=9)

for _, r in scale_rows.iterrows():
    family = all_llm_meta.get(r['llm'], {}).get('family', '')
    ax.annotate(f"({family})",
                xy=(r['scale_B'], -0.08), xycoords=('data', 'axes fraction'),
                ha='center', va='top', fontsize=7, color='#555555')

ax.set_xlabel('LLM Scale — approximate parameters', fontsize=11, labelpad=20)
ax.set_ylabel('Mean Jaccard Similarity\n(pipeline ∩ LLM / union)', fontsize=11)
ax.set_title(
    'Pipeline–LLM Entity Agreement Across Model Scales\n'
    '(no fixed gold standard · 183 articles · EN input · shading = ±1 SD)',
    fontsize=12)
ax.legend(title='Pipeline', fontsize=10)
ax.set_ylim(0, 1)
ax.yaxis.set_major_locator(ticker.MultipleLocator(0.1))
ax.grid(axis='y', alpha=0.3)
ax.grid(axis='x', alpha=0.15)

plt.tight_layout()
for ext in ['pdf', 'png']:
    fig.savefig(FIGURES_DIR / f'nb13_scaling_curve.{ext}',
                dpi=300 if ext == 'pdf' else 150, bbox_inches='tight')
plt.show()
print("✓ Figure saved: nb13_scaling_curve.pdf / .png")


In [ ]:
# ── CELL 11 : SAVE SUMMARY JSON ──────────────────────────────────────────────

flair_curve = jaccard_df[jaccard_df['pipeline'] == 'Flair'].sort_values('scale_B')
if not flair_curve.empty:
    peak        = flair_curve.loc[flair_curve['mean'].idxmax()]
    convergence = {'scale_B' : float(peak['scale_B']),
                   'jaccard' : round(float(peak['mean']), 4)}
else:
    convergence = None

model_run_status = {
    mid: {
        'n_ok'       : len([r for r in ckpt['results'] if r['status'] == 'ok']),
        'n_failed'   : len(ckpt['failed_ids']),
        'stop_reason': ckpt.get('stop_reason'),
    }
    for mid, ckpt in nb13_new_results.items()
}

summary = {
    'notebook'      : '13_scaling_study',
    'generated_at'  : datetime.now().isoformat(),
    'n_articles'    : len(ARTICLE_IDS_183),
    'note'          : ('Jaccard on (text, label) entity pairs — no fixed gold standard. '
                       'Old checkpoints (Qwen7B, Llama70B) used flat dict format. '
                       'GPT-OSS used merged system+user prompt. '
                       'Qwen3 used /no_think + 2048 token budget.'),
    'llm_systems'   : {
        mid: {
            'scale_B': all_llm_meta[mid].get('scale_B'),
            'family' : all_llm_meta[mid].get('family'),
            'n_used' : int(jaccard_df[jaccard_df['llm'] == mid]['n'].max())
                       if mid in jaccard_df['llm'].values else 0,
        }
        for mid in llm_order if mid in all_llm_meta
    },
    'jaccard_matrix': {
        f"{r['pipeline']}|{r['llm']}": round(r['mean'], 4)
        for _, r in jaccard_df.iterrows()
    },
    'convergence'   : convergence,
    'run_status'    : model_run_status,
}

with open(DATA_PROC / 'nb13_scaling_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("=" * 64)
print("NOTEBOOK 13 COMPLETE")
print("=" * 64)
print("  nb13_jaccard_matrix.csv")
print("  nb13_scaling_summary.json")
print("  nb13_scaling_curve.pdf / .png")
if convergence:
    print(f"\n  Flair convergence peak : ~{convergence['scale_B']}B  "
          f"(Jaccard = {convergence['jaccard']})")
print("\n  Run status:")
for mid, s in model_run_status.items():
    print(f"    {mid:<44} ok={s['n_ok']}  "
          f"fail={s['n_failed']}  [{s['stop_reason'] or 'not run'}]")
